# Paper authors — `author_list`, `team_size`, first and last author, countries

The Dimensions twin of `OpenAlex/notebook/paper_author.ipynb`, and a far cheaper one: the dump
carries `authors[]` on every publication row, in author order, with a resolved `researcher_id`
where Dimensions disambiguated the author and the affiliation addresses (ISO2 country) beside it.
`references_w_year.ipynb` already unnested that into `cache/pub_authors/`; this notebook
consolidates the parts, verifies them and writes one file.

## Semantics — the differences from OpenAlex, stated
| column | |
|---|---|
| `paper_id` | `pub.…` (the OpenAlex file keys on `work_id`; this one keys on `paper_id` like every sibling output here) |
| `author_list` | `;`-joined **researcher ids of the resolved author slots**, in author order. An unresolved author (no `researcher_id`) has nothing to list, so `len(author_list.split(';')) == n_resolved ≤ team_size` |
| `team_size` | every author slot, resolved or not — the honest head count |
| `n_resolved` | slots with a researcher id |
| `first_author`, `last_author` | researcher id of the first / last **slot**; null when that slot is unresolved (rather than silently promoting the next resolved author) |
| `corresponding_author` | the first slot flagged `corresponding` |
| `countries`, `n_countries` | distinct ISO2 codes over every author's `affiliations_address`, sorted; only 17–25 % of articles have any located author |

The OpenAlex file had to de-duplicate an author × affiliation table (17.8 % duplicate rows); the
Dimensions `authors[]` list is one entry per author slot, so there is nothing to de-duplicate —
verified below (a researcher id appearing twice in one list is counted and reported).

## Output
`Dimensions/output/paper_author.parquet`

In [1]:
import os, sys, time, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
OUT_FP = f'{dim.OUT}/paper_author.parquet'
assert dim.author_parts(), 'run notebook/references_w_year.ipynb first -- it writes cache/pub_authors/'
MEM = os.environ.get('NB_DUCKDB_MEM', '150GB')
con = duckdb.connect()
con.execute(f"SET memory_limit='{MEM}'")
con.execute(f"SET temp_directory='{dim.CACHE}/duckdb_tmp_authors'")
con.execute('SET preserve_insertion_order=false')
print(f'parts  : {len(dim.author_parts())} under {dim.AUTHORS}')
print(f'output : {OUT_FP}')

parts  : 4219 under /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_authors
output : /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_author.parquet


## 1. Build

In [2]:
%%time
t0 = time.time()
con.execute(f"""
COPY (
  SELECT pub_id AS paper_id, nullif(author_list, '') AS author_list, team_size::INTEGER AS team_size,
         n_resolved::INTEGER AS n_resolved, first_author, last_author, corresponding_author,
         nullif(countries, '') AS countries, n_countries::INTEGER AS n_countries
  FROM read_parquet('{dim.AUTHORS}/part_*.parquet')
  ORDER BY paper_id
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e9:.2f} GB) in {time.time()-t0:.0f}s')
print(f"  {con.execute(f'SELECT count(*) FROM read_parquet({OUT_FP!r})').fetchone()[0]:,} publications with at least one author slot")

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_author.parquet  (4.82 GB) in 103s
  142,431,518 publications with at least one author slot


## Verification

1. `paper_id` unique;
2. `n_resolved == len(author_list.split(';'))` wherever there is a list, and `n_resolved ≤ team_size`;
3. `first_author` / `last_author`, when not null, are the first / last entry of `author_list`
   **only if** the first / last slot is resolved — so the check is that they are *members* of the list;
4. duplicate researcher ids inside one list — counted, not asserted (a data fact about Dimensions).

In [3]:
%%time
chk = con.execute(f"""
WITH s AS (SELECT * FROM read_parquet('{OUT_FP}')),
     x AS (SELECT *, CASE WHEN author_list IS NULL THEN [] ELSE str_split(author_list, ';') END AS parts FROM s)
SELECT count(*)                                                            AS publications,
       count(*) - count(DISTINCT paper_id)                                 AS dup_paper_id,
       sum(team_size)                                                      AS author_slots,
       sum(n_resolved)                                                     AS resolved_slots,
       count(*) FILTER (WHERE n_resolved <> len(parts))                    AS size_mismatch,
       count(*) FILTER (WHERE n_resolved > team_size)                      AS resolved_gt_team,
       count(*) FILTER (WHERE first_author IS NOT NULL AND NOT list_contains(parts, first_author)) AS first_not_in_list,
       count(*) FILTER (WHERE last_author  IS NOT NULL AND NOT list_contains(parts, last_author))  AS last_not_in_list,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))     AS dup_in_list,
       count(*) FILTER (WHERE team_size = 1)                               AS solo,
       count(*) FILTER (WHERE countries IS NOT NULL)                       AS with_country
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<18} {int(v):>15,}')
ok = chk.dup_paper_id == 0 and chk.size_mismatch == 0 and chk.resolved_gt_team == 0 and chk.first_not_in_list == 0 and chk.last_not_in_list == 0
print(f"\n1-3. {'ALL PASS' if ok else 'FAILED -- see the counts above'}")
print(f'4.   lists with a repeated researcher id: {int(chk.dup_in_list):,}')
display(con.execute(f"""SELECT team_size, count(*) AS publications, round(avg(n_resolved),2) AS mean_resolved,
                        round(100.0*count(countries)/count(*),1) AS pct_with_country
                        FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY 1 LIMIT 12""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE n_resolved >= 3 AND countries IS NOT NULL LIMIT 5").fetchdf())
con.close()

  publications           142,431,518
  dup_paper_id                     0
  author_slots           496,565,750
  resolved_slots         383,014,165
  size_mismatch                    0
  resolved_gt_team                 0
  first_not_in_list                0
  last_not_in_list                 0
  dup_in_list                      0
  solo                    45,081,946
  with_country            86,405,175

1-3. ALL PASS
4.   lists with a repeated researcher id: 0


,team_size,publications,mean_resolved,pct_with_country
0,1,45081946,0.42,36.3
1,2,26652040,1.38,60.2
2,3,21121722,2.34,68.7
3,4,15266344,3.27,74.7
4,5,10738645,4.18,78.0
5,6,7626040,5.10,81.1
6,7,4849815,6.07,83.3
7,8,3333340,7.00,84.3
8,9,2199197,7.97,85.9
9,10,1639512,8.89,85.2


,paper_id,author_list,team_size,n_resolved,first_author,last_author,corresponding_author,countries,n_countries
0,pub.1004727675,ur.0617402577.15;ur.011463451235.61;ur.0655732...,6,5,ur.0617402577.15,ur.0622566755.51,ur.01107326057.37,DE,1
1,pub.1004727677,ur.01062205161.13;ur.0723232401.38;ur.01207025...,5,3,ur.01062205161.13,None,None,CH,1
2,pub.1004727681,ur.01221736263.00;ur.010235217662.66;ur.011574...,3,3,ur.01221736263.00,ur.011574434622.15,None,GB,1
3,pub.1004727682,ur.01024620600.77;ur.01363177263.44;ur.0637573...,4,4,ur.01024620600.77,ur.0705706363.18,ur.01024620600.77,CN,1
4,pub.1004727683,ur.01212320175.34;ur.067350221.02;ur.066477640...,3,3,ur.01212320175.34,ur.0664776406.25,None,SE,1
